In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
customer_rfm = pd.read_csv("../data/processed/rfm_data.csv")

In [4]:
customer_rfm.columns

Index(['CustomerID', 'Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score',
       'M_Score', 'RFM_Score', 'Segment'],
      dtype='object')

In [5]:
customer_rfm.head()

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
0,12346.0,325,1,77183.60,1,1,4,114,Lost
1,12347.0,1,7,4310.00,4,4,4,444,Champions
2,12348.0,74,4,1797.24,2,3,4,234,At Risk
3,12349.0,18,1,1757.55,3,1,4,314,Potential Loyalists
4,12350.0,309,1,334.40,1,1,2,112,Lost


In [7]:
customer_rfm['Churn'] = (customer_rfm['Recency'] > 90).astype(int)

In [8]:
customer_rfm['Churn'].value_counts()

Churn
0    2893
1    1445
Name: count, dtype: int64

In [18]:
(customer_rfm['Churn'].value_counts(normalize=True) * 100).round(2)

Churn
0    66.69
1    33.31
Name: proportion, dtype: float64

In [25]:

X = customer_rfm[['Frequency', 'Monetary']]
y = customer_rfm['Churn']




In [26]:
print(X.shape)
print(y.shape)

(4338, 2)
(4338,)


In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.20,random_state=True,stratify=y)


In [30]:

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier

models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, random_state=42)
    ),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, random_state=42
    ),
    "KNN": make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=5)
    ),
    "AdaBoost": AdaBoostClassifier(
        n_estimators=50, random_state=42
    )
}

for name, model in models.items():
    model.fit(X_train, y_train)
    


In [31]:

from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score

results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    by="F1 Score", ascending=False
).reset_index(drop=True)

results_df.round(4)


,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.7212,0.5786,0.5986,0.5884,0.7551
1,AdaBoost,0.6671,0.5000,0.6367,0.5601,0.7124
2,KNN,0.6751,0.5138,0.4498,0.4797,0.7044
3,Decision Tree,0.6636,0.4945,0.4637,0.4786,0.6126
4,Random Forest,0.6532,0.4776,0.4429,0.4596,0.6900


In [41]:

import joblib

model = models["Logistic Regression"]

joblib.dump(model, "../models/churn_logistic_regression.pkl")



['../models/churn_logistic_regression.pkl']